# TFDV Lab 1 — NYC Taxi Trip Data Validation

This notebook demonstrates **TensorFlow Data Validation (TFDV)** on the NYC Taxi Trips dataset.

**Modifications from original lab:**
- Dataset: NYC Taxi Trips (vs Census Income)
- Domain: Transportation (vs Socioeconomic)
- Anomalies: Negative fares, zero passengers, 24hr trips, unknown payment type, invalid RatecodeID
- Slicing: by `payment_type` and `VendorID` (vs `sex` and `race`)
- Structure: Modular `src/` package called from notebook

**Pipeline:**
1. Load and split data
2. Generate and visualize training statistics
3. Infer schema
4. Inject anomalies into eval set
5. Compare train vs eval statistics
6. Detect anomalies
7. Fix schema
8. Re-validate
9. Slice analysis by `payment_type` and `VendorID`

## Package Imports

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow_data_validation as tfdv
import pandas as pd

from src.data_loader      import load_data, split_data
from src.anomaly_injector import inject_anomalies
from src.statistics_utils import generate_statistics, compare_statistics
from src.schema_utils     import infer_schema, detect_anomalies, fix_schema, save_schema
from src.slicing_utils    import generate_sliced_statistics, compare_slices, list_slices

print(f'TFDV Version: {tfdv.__version__}')

/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWa

TFDV Version: 1.17.0


## Step 1: Load and Split Data

The NYC Taxi dataset contains trip records including fare, distance, duration, passenger count, payment type, and vendor information. We compute `trip_duration_seconds` as a derived feature from the pickup and dropoff timestamps.

In [2]:
df = load_data('data/nyc_taxi.csv')
train_df, eval_df = split_data(df, test_size=0.2, shuffle=False)
train_df.head()

[data_loader] Loaded 50000 rows | Columns: ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'payment_type', 'fare_amount', 'tip_amount', 'tolls_amount', 'total_amount', 'trip_duration_seconds']
[data_loader] Train: 40000 rows | Eval: 10000 rows


,VendorID,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,trip_duration_seconds
0,2,1.0,0.97,1.0,Cash,9.3,0.00,0.0,14.30,506
1,2,1.0,1.10,1.0,Credit,7.9,4.00,0.0,16.90,379
2,2,1.0,2.51,1.0,Credit,14.9,15.00,0.0,34.90,765
3,1,0.0,1.90,1.0,Credit,12.1,0.00,0.0,20.85,577
4,2,1.0,1.43,1.0,Credit,11.4,3.28,0.0,19.68,650


In [3]:
# Preview data types and basic stats
train_df.describe(include='all')

,VendorID,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,trip_duration_seconds
count,40000,40000.000000,40000.000000,40000,40000,40000.000000,40000.000000,40000.000000,40000.000000,40000.000000
unique,2,NaN,NaN,6,4,NaN,NaN,NaN,NaN,NaN
top,2,NaN,NaN,1.0,Credit,NaN,NaN,NaN,NaN,NaN
freq,29990,NaN,NaN,37395,29818,NaN,NaN,NaN,NaN,NaN
mean,NaN,1.545425,3.908028,NaN,NaN,20.051260,3.316440,0.579035,28.366169,944.519300
std,NaN,1.020423,4.550831,NaN,NaN,19.605358,4.319838,2.283980,23.693195,3392.617285
min,NaN,0.000000,0.000000,NaN,NaN,-346.000000,-0.010000,-20.750000,-351.000000,0.000000
25%,NaN,1.000000,1.220000,NaN,NaN,8.600000,0.000000,0.000000,15.120000,393.000000
50%,NaN,1.000000,2.230000,NaN,NaN,13.500000,2.520000,0.000000,21.000000,671.500000
75%,NaN,2.000000,4.560000,NaN,NaN,24.000000,4.400000,0.000000,32.280000,1089.000000


## Step 2: Generate and Visualize Training Statistics

TFDV computes descriptive statistics across all features. For numerical features: count, mean, std, min, max. For categorical: unique values, top values, missing rate.

In [4]:
train_stats = generate_statistics(train_df, dataset_name='TRAIN')

[statistics_utils] Generated statistics for 'TRAIN' (40000 rows)


In [5]:
tfdv.visualize_statistics(train_stats)

## Step 3: Infer Schema

TFDV infers a schema from training statistics. The schema captures:
- Feature types (INT, FLOAT, STRING)
- Presence (required vs optional)
- Domain (valid value ranges for numeric, valid values for categorical)

In [6]:
schema = infer_schema(train_stats)
save_schema(schema, 'schema/schema.pbtxt')

[schema_utils] Schema inferred from training statistics.


,Type,Presence,Valency,Domain
Feature name,,,,
'VendorID',BYTES,required,,-
'passenger_count',FLOAT,required,,-
'trip_distance',FLOAT,required,,-
'RatecodeID',STRING,required,,'RatecodeID'
'payment_type',STRING,required,,'payment_type'
'fare_amount',FLOAT,required,,-
'tip_amount',FLOAT,required,,-
'tolls_amount',FLOAT,required,,-
'total_amount',FLOAT,required,,-


,Values
Domain,
'RatecodeID',"'1.0', '2.0', '3.0', '4.0', '5.0', '99.0'"
'payment_type',"'Cash', 'Credit', 'Dispute', 'No charge'"


[schema_utils] Schema saved to schema/schema.pbtxt


## Step 4: Inject Anomalies into Eval Set

We manually inject 5 anomalous rows to simulate real-world data quality issues:

| Row | Anomaly | Type |
|-----|---------|------|
| 1 | `fare_amount = -15.0` | Negative fare |
| 2 | `passenger_count = 0` | Zero passengers |
| 3 | `trip_duration_seconds = 86400` | 24hr trip |
| 4 | `payment_type = 'CRYPTO'` | Unknown category |
| 5 | `RatecodeID = '99'` | Out-of-range value |

In [7]:
eval_df_dirty = inject_anomalies(eval_df)

# Preview the injected rows
eval_df_dirty.tail(6)

[anomaly_injector] Injected 5 anomalous rows → eval set now 10005 rows


,VendorID,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,trip_duration_seconds
9999,2,1.0,1.86,1.0,Credit,10.7,2.94,0.0,17.64,472
10000,1,2.0,3.50,1,Credit,-15.0,0.00,0.0,-15.00,720
10001,2,0.0,1.20,1,Cash,7.5,0.00,0.0,7.50,300
10002,1,1.0,2.00,1,Credit,10.0,2.00,0.0,12.00,86400
10003,2,3.0,5.00,1,CRYPTO,18.0,3.00,0.0,21.00,900
10004,1,1.0,0.50,99,Cash,5.0,0.00,0.0,5.00,180


## Step 5: Compare Train vs Eval Statistics

Visualize both datasets side by side. Look for distribution shifts in numeric features like `fare_amount` and `trip_duration_seconds` which should reveal the injected anomalies.

In [8]:
eval_stats_dirty = generate_statistics(eval_df_dirty, dataset_name='EVAL (dirty)')

[statistics_utils] Generated statistics for 'EVAL (dirty)' (10005 rows)


In [9]:
tfdv.visualize_statistics(
    lhs_statistics=eval_stats_dirty,
    rhs_statistics=train_stats,
    lhs_name='EVAL',
    rhs_name='TRAIN'
)

## Step 6: Detect Anomalies

Validate eval statistics against the training schema. TFDV should flag:
- `fare_amount` out of expected range
- `passenger_count` out of expected range  
- `trip_duration_seconds` outlier
- `payment_type` unexpected value ('CRYPTO')
- `RatecodeID` unexpected value ('99')

In [10]:
anomalies = detect_anomalies(eval_stats_dirty, schema)

,Anomaly short description,Anomaly long description,Anomaly types
Feature name,,,
'__index_level_0__',Column dropped,Column is completely missing,SCHEMA_MISSING_COLUMN
'RatecodeID',Unexpected string values,"Examples contain values missing from the schema: 1 (<1%), 6.0 (<1%), 99 (<1%).",ENUM_TYPE_UNEXPECTED_STRING_VALUES
'payment_type',Unexpected string values,Examples contain values missing from the schema: CRYPTO (<1%).,ENUM_TYPE_UNEXPECTED_STRING_VALUES


## Step 7: Fix Schema

Update the schema to handle valid edge cases:
- Set `fare_amount` min = 0.0
- Set `passenger_count` min = 1, max = 6
- Set `trip_duration_seconds` min = 30, max = 10800 (3 hours)
- Relax `payment_type` domain to 90% match
- Relax `RatecodeID` domain to 90% match

In [11]:
schema = fix_schema(schema)
save_schema(schema, 'schema/schema.pbtxt')

[schema_utils] Set fare_amount min=0.0
[schema_utils] Set passenger_count min=1, max=6
[schema_utils] Set trip_duration_seconds min=30, max=10800
[schema_utils] Relaxed payment_type domain to 90% match
[schema_utils] Relaxed RatecodeID domain to 90% match


,Type,Presence,Valency,Domain
Feature name,,,,
'VendorID',BYTES,required,,-
'passenger_count',FLOAT,required,,min: 1; max: 6
'trip_distance',FLOAT,required,,-
'RatecodeID',STRING,required,,'RatecodeID'
'payment_type',STRING,required,,'payment_type'
'fare_amount',FLOAT,required,,min: 0.000000; max: inf
'tip_amount',FLOAT,required,,-
'tolls_amount',FLOAT,required,,-
'total_amount',FLOAT,required,,-


,Values
Domain,
'RatecodeID',"'1.0', '2.0', '3.0', '4.0', '5.0', '99.0'"
'payment_type',"'Cash', 'Credit', 'Dispute', 'No charge'"


[schema_utils] Schema saved to schema/schema.pbtxt


## Step 8: Re-validate

After fixing the schema, re-validate the eval set. Anomalies for valid edge cases should be resolved.

In [12]:
updated_anomalies = detect_anomalies(eval_stats_dirty, schema)

,Anomaly short description,Anomaly long description,Anomaly types
Feature name,,,
'fare_amount',Out-of-range values,Unexpectedly low values: -70<0(upto six significant digits),FLOAT_TYPE_SMALL_FLOAT
'__index_level_0__',Column dropped,Column is completely missing,SCHEMA_MISSING_COLUMN
'passenger_count',The domain does not match the type,"The domain ""int_domain"" does not match the type: FLOAT",DOMAIN_INVALID_FOR_TYPE
'trip_duration_seconds',Multiple errors,Unexpectedly small value: 0. Unexpectedly large value: 86400.,INT_TYPE_SMALL_INT; INT_TYPE_BIG_INT


## Step 9: Slice Analysis

Analyze specific slices to understand if data quality and distributions vary across segments.

### 9a. Slice by Payment Type

Do Credit card trips differ from Cash trips in terms of fare, tip, and distance?

In [13]:
payment_slices = generate_sliced_statistics(
    df=train_df,
    schema=schema,
    slice_features={'payment_type': None},
    csv_path='slice_sample.csv'
)
list_slices(payment_slices)

[slicing_utils] Written 40000 rows to slice_sample.csv


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


[slicing_utils] Generated slices: ['All Examples', 'payment_type_Cash', 'payment_type_Credit', 'payment_type_Dispute', 'payment_type_No charge']
[slicing_utils] Available slices: ['All Examples', 'payment_type_Cash', 'payment_type_Credit', 'payment_type_Dispute', 'payment_type_No charge']


['All Examples',
 'payment_type_Cash',
 'payment_type_Credit',
 'payment_type_Dispute',
 'payment_type_No charge']

In [14]:
compare_slices(
    payment_slices,
    lhs_name='payment_type_Credit',
    rhs_name='payment_type_Cash'
)

[slicing_utils] Comparing slice 'payment_type_Credit' vs 'payment_type_Cash'


### 9b. Slice by VendorID

Do different taxi vendors report data consistently? Look for differences in distributions of numeric features like `trip_distance` and `fare_amount`.

In [15]:
vendor_slices = generate_sliced_statistics(
    df=train_df,
    schema=schema,
    slice_features={'VendorID': None},
    csv_path='slice_sample.csv'
)
list_slices(vendor_slices)

[slicing_utils] Written 40000 rows to slice_sample.csv
[slicing_utils] Generated slices: ['All Examples', 'VendorID_2', 'VendorID_1']
[slicing_utils] Available slices: ['All Examples', 'VendorID_2', 'VendorID_1']


['All Examples', 'VendorID_2', 'VendorID_1']

In [16]:
compare_slices(
    vendor_slices,
    lhs_name='VendorID_1',
    rhs_name='VendorID_2'
)

[slicing_utils] Comparing slice 'VendorID_1' vs 'VendorID_2'
